In [3]:
import os
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import pandas as pd
from torch.utils.data import DataLoader, Dataset, random_split
import GPUtil
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW, SGD, Adam
import numpy as np
from sklearn.model_selection import train_test_split

In [7]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_name)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [8]:
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'

dirty_df = pd.read_csv(dirty_path)
clean_df = pd.read_csv(clean_path)
clean_df.columns = dirty_df.columns

In [9]:
content_list = dirty_df.iloc[0, 1:-1].astype(str).values
name_list = dirty_df.columns.values
att_name_list = name_list[1: -1]
# print(name_list)
# print(a)
content = ""
test = zip(att_name_list, content_list)
for pair in test:
    content += pair[0] + ":" + pair[1] + ";"
print(content)

src:aa;flight:AA-3859-IAH-ORD;sched_dep_time:7:10 a.m.;act_dep_time:7:16 a.m.;sched_arr_time:9:40 a.m.;


In [10]:
prompt = "give me a prediction of the attribute 'act_arr_time'. Your answer should only be a value and nothing else. The original data is: " + content
print(prompt)

give me a prediction of the attribute 'act_arr_time'. Your answer should only be a value and nothing else. The original data is: src:aa;flight:AA-3859-IAH-ORD;sched_dep_time:7:10 a.m.;act_dep_time:7:16 a.m.;sched_arr_time:9:40 a.m.;


In [11]:
device = 'cuda:1'
model.to(device)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [12]:
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors='pt').to(device)
print(model_inputs)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  46430,    752,    264,
          19639,    315,    279,   7035,    364,    531,  11210,   3009,   4427,
           4615,   4226,   1265,   1172,    387,    264,    897,    323,   4302,
            770,     13,    576,   4024,    821,    374,     25,   2286,     25,
           5305,     26,  38390,     25,   6029,     12,     18,     23,     20,
             24,     12,   5863,     39,     12,   4276,  39414,   2397,  49258,
           3009,     25,     22,     25,     16,     15,    264,    744,  15642,
            531,  49258,   3009,     25,     22,     25,     16,     21,    264,
            744,  15642,  72243,  11210,   3009,     25,     24,     25,     19,
             15,    264,    744,  15642, 151645,    198, 151644,  77091,    198]],
       devic

In [13]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

0


In [46]:
test = content + "act_arr_time: ?"

In [49]:
input_ids = tokenizer([test], return_tensors='pt').to(device)
print(input_ids)
generated_ids = model.generate(
    **input_ids,
    max_new_tokens=512,
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

{'input_ids': tensor([[ 3548,    25,  5305,    26, 38390,    25,  6029,    12,    18,    23,
            20,    24,    12,  5863,    39,    12,  4276, 39414,  2397, 49258,
          3009,    25,    22,    25,    16,    15,   264,   744, 15642,   531,
         49258,  3009,    25,    22,    25,    16,    21,   264,   744, 15642,
         72243, 11210,  3009,    25,    24,    25,    19,    15,   264,   744,
         15642,   531, 11210,  3009,    25,   937]], device='cuda:1'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:1')}
859-IAH-ORD
Scheduling information:
- Departure time: 7:10 AM
- Arrival time: 7:16 AM
- Airline code: AA (Air France)
- Aircraft type: CF-2000

The aircraft departed at 7:10 AM. 

Therefore, the answer is 7:10 a.m. (noon). The aircraft departed at that specific hour. If you need any fur

In [4]:
torch.cuda.empty_cache()